# A · Masking and movement-change supervision

This experiment crosses the representation objective, mask policy and output loss. It asks whether predicting reference features helps preserve movement, and whether that answer changes when the encoder sees structured gaps in the observed sequence. Coordinate-pretrained and paired-JEPA encoders receive the same data, mask budgets and update counts. JEPA also changes the loss and adds its feature predictor, projector, regularizer and moving-average teacher, so this comparison evaluates the full declared pretraining recipe.

Read after notebooks 00–06. This notebook uses the prepared bundle and saved evaluation from that same run; the internal recipe group is `M`.


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Inspect the exact recipe cells

Compare coordinate versus JEPA within the same mask and output loss. Compare base versus paired-change supervision within the same encoder and mask. A comparison that changes both the encoder and the loss cannot isolate either factor. The three masks are whole-body time blocks, uniform joint-time tokens and connected anatomical regions over time.


In [ ]:
import pandas as pd
plan = study.artifact('plan.json')
group_recipes = [r for r in plan['recipes'] if r['group'] == 'M']
expected_count = 12 if plan.get('experiment_set', 'full') == 'full' else {'M': 4, 'T': 0, 'P': 2, 'I': 4, 'L': 0}['M']
assert len(group_recipes) == expected_count, 'Saved plan differs from the selected experiment set.'
recipe_ids = {r['recipe_id'] for r in group_recipes}
display(pd.DataFrame(group_recipes))
print('Final models:', len(group_recipes) * len(plan['seeds']), 'Seeds:', plan['seeds'])
if not group_recipes:
    print('This group is outside the saved core protocol. Its worked examples are educational; no results are implied.')


## Calculate the comparison within each person and training seed

Let $E_{p,s,e,m,l}$ be response error for person $p$, seed $s$, encoder $e$,
mask $m$, and output loss $l$. The gain from change supervision is
$G_{p,s,e,m}=E_{p,s,e,m,\mathrm{base}}-E_{p,s,e,m,\mathrm{change}}$.
Positive values favor change supervision. Keeping person, seed, encoder,
and mask fixed prevents their differences from entering this contrast.

The interaction $G_{\mathrm{JEPA}}-G_{\mathrm{coordinate}}$ asks whether
adding change supervision helps JEPA more than coordinate pretraining.
It does not estimate JEPA's overall advantage. We retain all masks and all
people below. These are descriptive development comparisons; notebook 06
explains paired uncertainty for the frozen primary comparison.


In [ ]:
import numpy as np
person_file = study.work / 'evaluation/per-person.csv'
if person_file.exists():
    person = pd.read_csv(person_file)
    design = pd.DataFrame(group_recipes).rename(columns={'recipe_id': 'method'})
    cells = person.merge(design, on='method', how='inner', validate='many_to_one')
    key = ['canonical_person_id', 'seed', 'encoder', 'pretraining_mask']
    errors = cells.pivot(index=key, columns='readout_or_training_objective', values='response_error')
    assert errors[['base', 'paired_change']].notna().all().all(), 'Report missing support before comparison.'
    gains = (errors['base'] - errors['paired_change']).rename('change_supervision_gain_deg')
    interaction = gains.unstack('encoder')
    interaction['JEPA_minus_coordinate_gain_deg'] = interaction['paired_jepa'] - interaction['coordinate']
    display(gains.reset_index())
    display(interaction.reset_index())
    print('Positive gain = lower response error; these rows reuse the same people.')
else:
    print('Run notebooks 04–05 to calculate these contrasts from retained predictions.')


The saved response error already gives equal weight to source families
within each person and excludes exact no-change duplicates from its primary
average. It includes penalties for failed predictions on eligible references.
Inspect coverage in the final table below before interpreting a small gain;
missing reference support can still limit what the population mean covers.


## Follow the shared dependencies

This tutorial inspects the existing central queue. It does not launch a separate copy of its group: that would duplicate pretraining and break the global budget. Notebook 04 launches all groups, and this table identifies the phases that belong to the present comparison.


In [ ]:
final_phases = [p for p in plan['phases'] if p['phase'] != 'pretrain' and p['recipe']['recipe_id'] in recipe_ids]
parent_ids = {parent for p in final_phases for parent in p['depends_on'] if parent != 'prepare'}
selected = [p for p in plan['phases'] if p in final_phases or p['phase_id'] in parent_ids]
display(pd.DataFrame([{'phase_id': p['phase_id'], 'phase': p['phase'], 'seed': p['seed'],
                      'depends_on': ', '.join(p['depends_on'])} for p in selected]))


## Inspect completed checkpoints and learning histories

Each completed phase links its retained checkpoint, history and predictions to its source identity. Missing phases are reported as pending; a checkpoint from another study is not substituted.


In [ ]:
ledger_path = study.work / 'ledger.json'
completed = json.loads(ledger_path.read_text()).get('completed', {}) if ledger_path.exists() else {}
rows = []
for phase in selected:
    saved = completed.get(phase['phase_id'])
    result = saved.get('result', {}) if saved else {}
    rows.append({'phase_id': phase['phase_id'], 'status': 'complete' if saved else 'pending',
                 'checkpoint': result.get('checkpoint'), 'predictions': result.get('predictions')})
display(pd.DataFrame(rows))
history_candidates = []
for row in rows:
    if row['checkpoint']:
        history_path = Path(row['checkpoint']).parent / 'history.json'
        if history_path.exists(): history_candidates.append(history_path)
if history_candidates:
    history_path = history_candidates[0]
    print('First declared completed history:', history_path)
    display(pd.DataFrame(json.loads(history_path.read_text())))
else:
    print('No completed histories yet. Run or resume the central queue from notebook 04.')


## Read group results on the common population

Keep both endpoints in every training batch with equal coordinate exposure. Read the change term together with per-example and valid re-pairing controls from experiment E. Improvement limited to the optimized knee measurement needs the prespecified untrained measurement or held conditions before a broader preservation claim.


In [ ]:
person_path = study.work / 'evaluation/per-person.csv'
if person_path.exists():
    people = pd.read_csv(person_path)
    display(people.loc[people['method'].isin(recipe_ids)])
    coverage_path = study.work / 'evaluation/coverage.csv'
    if coverage_path.exists():
        coverage = pd.read_csv(coverage_path)
        display(coverage.loc[coverage['method'].isin(recipe_ids)])
else:
    print('Evaluation is pending. These recipe cards do not fabricate or extrapolate results.')


Read the matching controls from the other experiment tutorials before attribution. Full evaluation and numerical reconstruction are covered by notebooks 05 and 06.
